In [2]:
!pip install -U sentence-transformers

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import pandas as pd
import torch

# Check if GPU is available (Colab usually provides a T4)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 1. Load the dataset you created
df = pd.read_csv('final_real_world_dataset.csv')

# 2. Split into Train (90%) and Test (10%)
train_df = df.sample(frac=0.9, random_state=42)
test_df = df.drop(train_df.index)

# 3. Convert to InputExample objects
train_examples = [
    InputExample(texts=[row['anchor'], row['positive'], row['negative']])
    for _, row in train_df.iterrows()
]

# 4. Create DataLoader (Batch size 16 is safe for Colab Free Tier)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Load a pre-trained Transformer model
model = SentenceTransformer('sentence-transformers/all-distilroberta-v1', device=device)

# FIX: Changed 'margin' to 'triplet_margin'
train_loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=0.5
)

sentences1 = test_df['anchor'].tolist()
sentences2 = test_df['positive'].tolist()
sentences3 = test_df['negative'].tolist()

# This evaluator checks if Anchor is closer to Positive than to Negative
evaluator = evaluation.TripletEvaluator(sentences1, sentences2, sentences3)

print("🚀 Starting Fine-Tuning...")

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=1,           # 1-2 epochs is usually enough for fine-tuning
    evaluation_steps=100,
    warmup_steps=100,
    output_path='volunteer_model_v1'
)

print("✅ Training Complete! Model saved to folder: 'volunteer_model_v1'")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training Complete! Model saved to folder: 'volunteer_model_v1'


In [3]:
import shutil

# 1. Zip the folder
shutil.make_archive('volunteer_model_v1_download', 'zip', 'volunteer_model_v1')

# 2. Trigger the download to your computer
from google.colab import files
files.download('volunteer_model_v1_download.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>